# Phase-Matching Table

Builds on `emode_helpers.py` (session/geometry/mode-tracking helpers factored out of `AlN_2D_4_sweep.ipynb`).
See `EMode_Troubleshooting_Log.md` for the roadmap and the backend quirks these helpers work around.

In [1]:
import numpy as np
import emode_helpers as eh

wavelength_fund = 450.0
w_core = 400.0  # [nm]
h_core = 335.0  # [nm]
dx, dy = 10.0, 10.0

eq_o = '(1+2.8032/(1-0.015287/x**2)+0.36335/(1-0.036095/x**2)-33508000/(1+367200000/x**2))**0.5'
eq_e = '(1+0.017061/(1-0.043855/x**2)+3.1976/(1-0.022642/x**2)-57269000/(1-74226000/x**2))**0.5'
anisotropic_equation = f"[{eq_o},{eq_e},{eq_o}]"

In [2]:
h_core = 350.0

## Regression check

Re-runs the mode-tracking test already validated in `AlN_2D_4_sweep.ipynb` (widths 395/400, TM04 tracked via overlap), using the new `emode_helpers` functions instead of inline code -- confirms the refactor didn't change behavior before building further on top of it.

In [2]:
test_widths = [395, 400]
Nmodes_shg = 35

ref_width = 400
ref_mode_idx = 30
results = []

for w in test_widths:
    print(f"\n--- Testing width: {w} nm (relaunching fresh EMode session) ---")
    try:
        em = eh.launch_session('phase_matching_table')
        eh.setup_waveguide(em, anisotropic_equation, h_core, ref_width)

        # Re-solve the reference width fresh (previous session's labeled profile no longer exists)
        eh.solve_modes(em, 'ref', wavelength=wavelength_fund/2, num_modes=Nmodes_shg,
                       x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

        eh.set_core_width(em, h_core, w)
        eh.solve_modes(em, 'cur', wavelength=wavelength_fund/2, num_modes=Nmodes_shg,
                       x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

        best_idx, best_overlap, overlaps = eh.track_mode(em, 'ref', ref_mode_idx, 'cur', Nmodes_shg)
        print(f"  Checked modes {list(overlaps.keys())}")
        print(f"  Best match: mode {best_idx}, overlap = {best_overlap:.4f}")
        if best_overlap < 0.8:
            print(f"  WARNING: low overlap ({best_overlap:.3f}) - true match may be outside the search window.")

        results.append((w, best_idx, best_overlap))
        em.close(save=False)
        ref_width, ref_mode_idx = w, best_idx

    except Exception as e:
        print(f"  FAILED at width {w}: {repr(e)}")
        results.append((w, None, None))

print("\nWidth (nm) | Best Match Mode | Overlap")
for w, idx, ov in results:
    print(f"{w:<11}| {str(idx):<17}| {ov if ov is None else f'{ov:.4f}'}")


--- Testing width: 395 nm (relaunching fresh EMode session) ---
  Checked modes [25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
  Best match: mode 30, overlap = 0.9997

--- Testing width: 400 nm (relaunching fresh EMode session) ---
  Checked modes [25, 26, 27, 28, 29, 30, 31, 32, 33, 34]
  Best match: mode 30, overlap = 0.9998

Width (nm) | Best Match Mode | Overlap
395        | 30               | 0.9997
400        | 30               | 0.9998


## Smoke test: `find_crossing()` for TM00 vs TM04

Cheap relative to the gradient functions -- one session, two `sweep()` calls, no extra `overlap()`
calls. Uses the confirmed (h_core=335, w_core=400) point where mode 30 = TM04.

In [ ]:
import importlib
importlib.reload(eh)

<module 'emode_helpers' from 'c:\\Users\\brent\\Documents\\Github Repositories\\eMode_scripts\\emode_helpers.py'>

In [3]:
wavelengths_shg = np.arange(224, 232, 1.0)

crossing = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg,
                             fund_mode_idx=0, target_mode_idx=30,
                             num_modes_fund=2, num_modes_target=35,
                             x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing is None:
    print(f"No crossing found in {wavelengths_shg[0]}-{wavelengths_shg[-1]}nm -- widen the window and retry.")
else:
    print(f"Phase-matching wavelength (SHG side): {crossing['wavelength_shg']:.3f} nm")
    print(f"n_eff at crossing: {crossing['n_eff']:.5f}")
    print(f"dn/dWL (fundamental): {crossing['dn_dWL_fund']:.6e} per nm")
    print(f"dn/dWL (TM04):        {crossing['dn_dWL_target']:.6e} per nm")

Phase-matching wavelength (SHG side): 225.080 nm
n_eff at crossing: 2.05049
dn/dWL (fundamental): -1.620134e-03 per nm
dn/dWL (TM04):        -1.498068e-02 per nm


## Sanity check: `mode_gradient_width_height` on the fundamental mode only

Fundamental is cheap (num_modes=6 vs. 35 for TM04), so this is a fast partial test of the gradient
machinery -- 4 `solve_and_identify()` calls at low mode count -- before committing to the full
~45-60 min run across both modes.

In [9]:
fund_wavelength = 2 * crossing['wavelength_shg']
fund_base_point = (h_core, w_core, fund_wavelength)

fund_grad = eh.mode_gradient_width_height(anisotropic_equation, fund_base_point, 0, num_modes=6,
                                           x_resolution=dx/2, y_resolution=dy/2)

print(f"Fundamental base point: h={h_core}, w={w_core}, wavelength={fund_wavelength:.3f} nm")
print(f"dn/dwidth:  {fund_grad['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {fund_grad['dn_dheight']:.6e} per nm")

Fundamental base point: h=335.0, w=400.0, wavelength=456.488 nm
dn/dwidth:  2.503775e-04 per nm
dn/dheight: 3.804437e-04 per nm


## Full run: `mode_gradient_width_height` for TM04

Expensive piece (num_modes=35, ~25-30 min estimated). Same anchor point (h=335, w=400) but at
TM04's own crossing wavelength, not the fundamental's 2x wavelength.

In [10]:
target_base_point = (h_core, w_core, crossing['wavelength_shg'])

target_grad = eh.mode_gradient_width_height(anisotropic_equation, target_base_point, 30, num_modes=35,
                                             x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print(f"TM04 base point: h={h_core}, w={w_core}, wavelength={crossing['wavelength_shg']:.3f} nm")
print(f"dn/dwidth:  {target_grad['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {target_grad['dn_dheight']:.6e} per nm")

TM04 base point: h=335.0, w=400.0, wavelength=228.244 nm
dn/dwidth:  1.067646e-04 per nm
dn/dheight: 3.172773e-03 per nm


## Combine: implicit-function wavelength gradient

Completes the first full table row (TM00 vs TM04, h=335, w=400) -- pure arithmetic, no EMode calls.

In [11]:
wl_grad = eh.implicit_wavelength_gradient(fund_grad, target_grad, crossing)

print("=== TM00 vs TM04 phase-matching table row (h=335, w=400) ===")
print(f"Phase-matching wavelength (SHG side): {crossing['wavelength_shg']:.3f} nm")
print(f"n_eff at crossing: {crossing['n_eff']:.5f}")
print(f"dn/dWL   -- fund: {crossing['dn_dWL_fund']:.6e}/nm   TM04: {crossing['dn_dWL_target']:.6e}/nm")
print(f"dn/dwidth -- fund: {fund_grad['dn_dwidth']:.6e}/nm   TM04: {target_grad['dn_dwidth']:.6e}/nm")
print(f"dn/dheight-- fund: {fund_grad['dn_dheight']:.6e}/nm   TM04: {target_grad['dn_dheight']:.6e}/nm")
print(f"d(PM wavelength)/d(width):  {wl_grad['dWL_dwidth']:.6f} nm per nm")
print(f"d(PM wavelength)/d(height): {wl_grad['dWL_dheight']:.6f} nm per nm")

=== TM00 vs TM04 phase-matching table row (h=335, w=400) ===
Phase-matching wavelength (SHG side): 228.244 nm
n_eff at crossing: 2.05082
dn/dWL   -- fund: -1.544411e-03/nm   TM04: -1.359859e-02/nm
dn/dwidth -- fund: 2.503775e-04/nm   TM04: 1.067646e-04/nm
dn/dheight-- fund: 3.804437e-04/nm   TM04: 3.172773e-03/nm
d(PM wavelength)/d(width):  -0.011914 nm per nm
d(PM wavelength)/d(height): 0.231648 nm per nm


## Identify TM40 / TM02 / TM20

Solve the same 35-mode anchor point once and print `em.report()` (n_eff, TE-fraction, confinement,
etc. for every mode) to triage candidates before plotting anything -- much cheaper than blindly
plotting all 35 mode profiles one at a time.

In [12]:
em = eh.launch_session('identify_modes', verbose=True)
eh.setup_waveguide(em, anisotropic_equation, h_core, w_core)
eh.solve_modes(em, 'anchor', wavelength=crossing['wavelength_shg'], num_modes=35,
               x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
em.report()

{'_default': {'Mode #': ['TM-0', 2.570712018753997, '0.0 %', '0.000'],
  'n_eff': ['TM-1', 2.524106365118906, '0.1 %', '0.000'],
  'TE %': ['TM-2', 2.5028102836526265, '0.0 %', '0.000'],
  'Loss (dB/m)': ['TM-3', 2.4546581886101007, '0.5 %', '0.000']}}

In [15]:
fdm_result = em.FDM()
print("TM_indices:", fdm_result['TM_indices'])
print("TE_fraction (all modes):", fdm_result['TE_fraction'])


: 

## Retry: identify modes via TE_fraction/TM_indices (fresh session)

Previous attempt crashed the kernel on a second `FDM()` call in the same long-lived session (see
`EMode_Troubleshooting_Log.md`). This time: fresh session, capture `solve_modes()`'s return
directly (no redundant re-solve), and set the debug env var EMode support asked for.

In [3]:
import os
os.environ['EMODE_LOG_SECRET'] = 'hAoai*9gogZU'  # don't commit the real value to git

em = eh.launch_session('identify_modes_v2', verbose=True)
eh.setup_waveguide(em, anisotropic_equation, h_core, w_core)
fdm_result = eh.solve_modes(em, 'anchor', wavelength=crossing['wavelength_shg'], num_modes=35,
                             x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
em.close(save=False)

print("TM_indices:", fdm_result['TM_indices'])
print("TE_fraction (all modes):", fdm_result['TE_fraction'])

TM_indices: [ 0  1  2  3  4  6  9 10 11 15 16 19 20 21 24 26 29 30 31]
TE_fraction (all modes): [1.16873418e-04 6.32104998e-04 4.43504229e-04 3.33490598e-03
 2.32861800e-03 9.99375627e-01 3.51630353e-03 9.80931315e-01
 9.98994256e-01 1.97664779e-02 5.24234498e-03 7.93035582e-03
 9.96938488e-01 9.91045454e-01 9.94598534e-01 2.94125361e-02
 1.85736023e-02 9.85947849e-01 9.27060687e-01 1.04482877e-01
 2.69022235e-02 1.85766143e-02 9.72409630e-01 9.70420774e-01
 3.02398365e-01 7.45799281e-01 1.68188925e-01 8.12138837e-01
 8.37231321e-01 2.34560914e-01 9.12187188e-03 2.44322148e-01
 6.66343807e-01 9.22826271e-01 8.98054226e-01]


In [4]:
spot_check_width = 350
anchor_point = (h_core, w_core, crossing['wavelength_shg'])
id_result = eh.solve_and_identify(anisotropic_equation, anchor_point, 30,
                                   (h_core, spot_check_width, crossing['wavelength_shg']),
                                   num_modes=35, x_resolution=dx/2, y_resolution=dy/2,
                                   max_effective_index=2.6)
print(f"At width={spot_check_width}nm: TM04 tracked to mode index {id_result['mode_idx']}, overlap={id_result['overlap']:.4f}")


At width=350nm: TM04 tracked to mode index 28, overlap=0.9385


In [5]:
wavelengths_shg_wide = np.arange(210, 251, 2.0)
crossing_350 = eh.find_crossing(anisotropic_equation, h_core, spot_check_width, wavelengths_shg_wide,
                                 fund_mode_idx=0, target_mode_idx=id_result['mode_idx'],
                                 num_modes_fund=2, num_modes_target=35,
                                 x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing_350 is None:
    print(f"No crossing found in {wavelengths_shg_wide[0]}-{wavelengths_shg_wide[-1]}nm at width={spot_check_width}nm.")
else:
    predicted_wl = crossing['wavelength_shg'] + (-0.011914) * (spot_check_width - w_core)  # known dWL/dwidth from the h=335,w=400 row
    print(f"Brute-force crossing at width={spot_check_width}nm: {crossing_350['wavelength_shg']:.3f} nm")
    print(f"Linear model prediction:                         {predicted_wl:.3f} nm")
    print(f"Difference: {crossing_350['wavelength_shg'] - predicted_wl:.3f} nm")


Brute-force crossing at width=350nm: 225.704 nm
Linear model prediction:                         225.676 nm
Difference: 0.029 nm


In [7]:
em = eh.launch_session('mode_survey', verbose=True)
eh.setup_waveguide(em, anisotropic_equation, h_core, w_core)
eh.solve_modes(em, 'anchor', wavelength=crossing['wavelength_shg'], num_modes=35,
               x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
em.plot(mode=0, component='Ey')

'ok'

In [ ]:
em.plot(mode=2, component='Ey')

## Switch to h=350 (the real target height)

h=335 was always a validation stand-in; the linear model's spot-check at width=350 already
validated the methodology well, so rather than redo a "clean" h=335 row, move straight to the real
target. Track TM04 from the confirmed h=335 anchor to h=350 via `solve_and_identify` (not a blind
re-guess) before doing anything else there. No kernel restart needed -- just close any open `em`
session (handled automatically by `clear='all'` on the next launch) and update `h_core`.

In [ ]:
anchor_h = 335.0  # previously validated anchor height (mode 30 = TM04, confirmed clean)
h_core = 350.0    # switching to the real target height

id_result_h350 = eh.solve_and_identify(anisotropic_equation, (anchor_h, w_core, crossing['wavelength_shg']), 30,
                                        (h_core, w_core, crossing['wavelength_shg']),
                                        num_modes=35, x_resolution=dx/2, y_resolution=dy/2,
                                        max_effective_index=2.6)
print(f"At h={h_core}nm: TM04 tracked to mode index {id_result_h350['mode_idx']}, overlap={id_result_h350['overlap']:.4f}")

NameError: name 'crossing' is not defined

## `find_crossing` at h=350 (real target height + height-axis spot-check)

Serves double duty: builds toward the real h=350 table row, and spot-checks the linear model along
the height axis (more sensitive than width, so a better stress-test than the w=350 check).

In [ ]:
wavelengths_shg_h350 = np.arange(224, 234, 1.0)
crossing_h350 = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg_h350,
                                  fund_mode_idx=0, target_mode_idx=30,
                                  num_modes_fund=2, num_modes_target=35,
                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing_h350 is None:
    print(f"No crossing found in {wavelengths_shg_h350[0]}-{wavelengths_shg_h350[-1]}nm at h=350nm.")
else:
    predicted_wl = 225.080 + 0.231648 * (h_core - anchor_h)  # known dWL/dheight from the h=335,w=400 row
    print(f"Brute-force crossing at h=350nm: {crossing_h350['wavelength_shg']:.3f} nm")
    print(f"Linear model prediction:        {predicted_wl:.3f} nm")
    print(f"Difference: {crossing_h350['wavelength_shg'] - predicted_wl:.3f} nm")

## Fundamental + TM04 gradients at h=350 (the real table row)

Mirrors the h=335 gradient cells, now at the real target height. Fundamental first (cheap, num_modes=6,
~5-10 min), then TM04 (expensive, num_modes=35, ~25-30 min based on the h=335 precedent).

In [6]:
fund_wavelength_h350 = 2 * crossing_h350['wavelength_shg']
fund_base_point_h350 = (h_core, w_core, fund_wavelength_h350)

fund_grad_h350 = eh.mode_gradient_width_height(anisotropic_equation, fund_base_point_h350, 0, num_modes=6,
                                                x_resolution=dx/2, y_resolution=dy/2)

print(f"Fundamental base point: h={h_core}, w={w_core}, wavelength={fund_wavelength_h350:.3f} nm")
print(f"dn/dwidth:  {fund_grad_h350['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {fund_grad_h350['dn_dheight']:.6e} per nm")

Fundamental base point: h=350.0, w=400.0, wavelength=456.488 nm
dn/dwidth:  2.524087e-04 per nm
dn/dheight: 3.398009e-04 per nm


In [7]:
target_base_point_h350 = (h_core, w_core, crossing_h350['wavelength_shg'])

target_grad_h350 = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_h350, 30, num_modes=35,
                                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print(f"TM04 base point: h={h_core}, w={w_core}, wavelength={crossing_h350['wavelength_shg']:.3f} nm")
print(f"dn/dwidth:  {target_grad_h350['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {target_grad_h350['dn_dheight']:.6e} per nm")

TM04 base point: h=350.0, w=400.0, wavelength=228.244 nm
dn/dwidth:  1.086564e-04 per nm
dn/dheight: 2.892175e-03 per nm


In [8]:
wl_grad_h350 = eh.implicit_wavelength_gradient(fund_grad_h350, target_grad_h350, crossing_h350)

print("=== TM00 vs TM04 phase-matching table row (h=350, w=400) ===")
print(f"Phase-matching wavelength (SHG side): {crossing_h350['wavelength_shg']:.3f} nm")
print(f"n_eff at crossing: {crossing_h350['n_eff']:.5f}")
print(f"dn/dWL   -- fund: {crossing_h350['dn_dWL_fund']:.6e}/nm   TM04: {crossing_h350['dn_dWL_target']:.6e}/nm")
print(f"dn/dwidth -- fund: {fund_grad_h350['dn_dwidth']:.6e}/nm   TM04: {target_grad_h350['dn_dwidth']:.6e}/nm")
print(f"dn/dheight-- fund: {fund_grad_h350['dn_dheight']:.6e}/nm   TM04: {target_grad_h350['dn_dheight']:.6e}/nm")
print(f"d(PM wavelength)/d(width):  {wl_grad_h350['dWL_dwidth']:.6f} nm per nm")
print(f"d(PM wavelength)/d(height): {wl_grad_h350['dWL_dheight']:.6f} nm per nm")

=== TM00 vs TM04 phase-matching table row (h=350, w=400) ===
Phase-matching wavelength (SHG side): 228.244 nm
n_eff at crossing: 2.05082
dn/dWL   -- fund: -1.544411e-03/nm   TM04: -1.359859e-02/nm
dn/dwidth -- fund: 2.524087e-04/nm   TM04: 1.086564e-04/nm
dn/dheight-- fund: 3.398009e-04/nm   TM04: 2.892175e-03/nm
d(PM wavelength)/d(width):  -0.011926 nm per nm
d(PM wavelength)/d(height): 0.211742 nm per nm


## Resume mode survey at h=350 (TM40 / TM02 / TM20)

Fresh session at the real target height. Single solve captures TE_fraction/TM_indices AND stays
open for plotting -- calling FDM() a second time in the same session is what crashed the kernel
before, so this time we get everything from one solve.

In [9]:
em = eh.launch_session('mode_survey_h350', verbose=True)
eh.setup_waveguide(em, anisotropic_equation, h_core, w_core)
fdm_result_h350 = eh.solve_modes(em, 'anchor', wavelength=crossing_h350['wavelength_shg'], num_modes=35,
                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print("TM_indices:", fdm_result_h350['TM_indices'])
print("TE_fraction (all modes):", fdm_result_h350['TE_fraction'])

TM_indices: [ 0  1  2  3  4  6  9 10 12 15 16 17 20 21 25 27 28 30]
TE_fraction (all modes): [1.16079971e-04 6.49378069e-04 4.36490979e-04 3.43128227e-03
 2.53433543e-03 9.99330486e-01 1.18263807e-03 9.99031000e-01
 9.83739313e-01 1.89284057e-02 5.24368507e-03 9.96542484e-01
 9.33405729e-03 9.97898956e-01 9.89168769e-01 2.00781848e-02
 3.25046405e-02 2.03687716e-02 9.74845246e-01 9.80317284e-01
 1.51489462e-02 3.99445455e-02 9.89762837e-01 9.55406093e-01
 5.29325861e-01 4.92534305e-01 8.46200479e-01 2.43553690e-01
 3.93589853e-01 5.27575424e-01 9.97173924e-03 8.18012513e-01
 7.39070608e-01 6.26804903e-01 9.29344726e-01]


In [11]:
em.plot(mode=3, component='Ey')

'ok'

In [17]:
em.plot(mode=21, component='Ey')



'ok'

In [18]:
## === TM00 vs TM20 (mode 4) ===

wavelengths_shg_TM20 = np.arange(210, 251, 2.0)
Nmodes_TM20 = 12  # index 4 + window(5) + buffer, vs. 35 for TM04 -- much cheaper

crossing_TM20 = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg_TM20,
                                  fund_mode_idx=0, target_mode_idx=4,
                                  num_modes_fund=2, num_modes_target=Nmodes_TM20,
                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing_TM20 is None:
    print(f"No crossing found for TM20 in {wavelengths_shg_TM20[0]}-{wavelengths_shg_TM20[-1]}nm.")
else:
    print(f"TM20 crossing: {crossing_TM20['wavelength_shg']:.3f} nm, n_eff={crossing_TM20['n_eff']:.5f}")
    print(f"dn/dWL -- fund: {crossing_TM20['dn_dWL_fund']:.6e}/nm   TM20: {crossing_TM20['dn_dWL_target']:.6e}/nm")

No crossing found for TM20 in 210.0-250.0nm.


In [20]:
if crossing_TM20 is None:
    print("Skipping TM20 gradients -- no crossing found in range.")
    fund_grad_TM20 = None
else:
    fund_wavelength_TM20 = 2 * crossing_TM20['wavelength_shg']
    fund_base_point_TM20 = (h_core, w_core, fund_wavelength_TM20)

    fund_grad_TM20 = eh.mode_gradient_width_height(anisotropic_equation, fund_base_point_TM20, 0, num_modes=6,
                                                    x_resolution=dx/2, y_resolution=dy/2)

    print(f"Fundamental base point: h={h_core}, w={w_core}, wavelength={fund_wavelength_TM20:.3f} nm")
    print(f"dn/dwidth:  {fund_grad_TM20['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {fund_grad_TM20['dn_dheight']:.6e} per nm")

Skipping TM20 gradients -- no crossing found in range.


In [21]:
if crossing_TM20 is None:
    print("Skipping TM20 target gradient -- no crossing found in range.")
    target_grad_TM20 = None
else:
    target_base_point_TM20 = (h_core, w_core, crossing_TM20['wavelength_shg'])

    target_grad_TM20 = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_TM20, 4, num_modes=Nmodes_TM20,
                                                      x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

    print(f"TM20 base point: h={h_core}, w={w_core}, wavelength={crossing_TM20['wavelength_shg']:.3f} nm")
    print(f"dn/dwidth:  {target_grad_TM20['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {target_grad_TM20['dn_dheight']:.6e} per nm")

Skipping TM20 target gradient -- no crossing found in range.


In [22]:
if crossing_TM20 is None:
    print("TM00 vs TM20: no crossing found in range -- excluded from table.")
    wl_grad_TM20 = None
else:
    wl_grad_TM20 = eh.implicit_wavelength_gradient(fund_grad_TM20, target_grad_TM20, crossing_TM20)

    print("=== TM00 vs TM20 phase-matching table row (h=350, w=400) ===")
    print(f"Phase-matching wavelength (SHG side): {crossing_TM20['wavelength_shg']:.3f} nm")
    print(f"n_eff at crossing: {crossing_TM20['n_eff']:.5f}")
    print(f"dn/dWL   -- fund: {crossing_TM20['dn_dWL_fund']:.6e}/nm   TM20: {crossing_TM20['dn_dWL_target']:.6e}/nm")
    print(f"dn/dwidth -- fund: {fund_grad_TM20['dn_dwidth']:.6e}/nm   TM20: {target_grad_TM20['dn_dwidth']:.6e}/nm")
    print(f"dn/dheight-- fund: {fund_grad_TM20['dn_dheight']:.6e}/nm   TM20: {target_grad_TM20['dn_dheight']:.6e}/nm")
    print(f"d(PM wavelength)/d(width):  {wl_grad_TM20['dWL_dwidth']:.6f} nm per nm")
    print(f"d(PM wavelength)/d(height): {wl_grad_TM20['dWL_dheight']:.6f} nm per nm")

TM00 vs TM20: no crossing found in range -- excluded from table.


In [23]:
## === TM00 vs TM02 (mode 6) ===

wavelengths_shg_TM02 = np.arange(210, 251, 2.0)
Nmodes_TM02 = 14  # index 6 + window(5) + buffer

crossing_TM02 = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg_TM02,
                                  fund_mode_idx=0, target_mode_idx=6,
                                  num_modes_fund=2, num_modes_target=Nmodes_TM02,
                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing_TM02 is None:
    print(f"No crossing found for TM02 in {wavelengths_shg_TM02[0]}-{wavelengths_shg_TM02[-1]}nm.")
else:
    print(f"TM02 crossing: {crossing_TM02['wavelength_shg']:.3f} nm, n_eff={crossing_TM02['n_eff']:.5f}")
    print(f"dn/dWL -- fund: {crossing_TM02['dn_dWL_fund']:.6e}/nm   TM02: {crossing_TM02['dn_dWL_target']:.6e}/nm")

No crossing found for TM02 in 210.0-250.0nm.


In [24]:
if crossing_TM02 is None:
    print("Skipping TM02 gradients -- no crossing found in range.")
    fund_grad_TM02 = None
else:
    fund_wavelength_TM02 = 2 * crossing_TM02['wavelength_shg']
    fund_base_point_TM02 = (h_core, w_core, fund_wavelength_TM02)

    fund_grad_TM02 = eh.mode_gradient_width_height(anisotropic_equation, fund_base_point_TM02, 0, num_modes=6,
                                                    x_resolution=dx/2, y_resolution=dy/2)

    print(f"Fundamental base point: h={h_core}, w={w_core}, wavelength={fund_wavelength_TM02:.3f} nm")
    print(f"dn/dwidth:  {fund_grad_TM02['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {fund_grad_TM02['dn_dheight']:.6e} per nm")

Skipping TM02 gradients -- no crossing found in range.


In [25]:
if crossing_TM02 is None:
    print("Skipping TM02 target gradient -- no crossing found in range.")
    target_grad_TM02 = None
else:
    target_base_point_TM02 = (h_core, w_core, crossing_TM02['wavelength_shg'])

    target_grad_TM02 = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_TM02, 6, num_modes=Nmodes_TM02,
                                                      x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

    print(f"TM02 base point: h={h_core}, w={w_core}, wavelength={crossing_TM02['wavelength_shg']:.3f} nm")
    print(f"dn/dwidth:  {target_grad_TM02['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {target_grad_TM02['dn_dheight']:.6e} per nm")

Skipping TM02 target gradient -- no crossing found in range.


In [26]:
if crossing_TM02 is None:
    print("TM00 vs TM02: no crossing found in range -- excluded from table.")
    wl_grad_TM02 = None
else:
    wl_grad_TM02 = eh.implicit_wavelength_gradient(fund_grad_TM02, target_grad_TM02, crossing_TM02)

    print("=== TM00 vs TM02 phase-matching table row (h=350, w=400) ===")
    print(f"Phase-matching wavelength (SHG side): {crossing_TM02['wavelength_shg']:.3f} nm")
    print(f"n_eff at crossing: {crossing_TM02['n_eff']:.5f}")
    print(f"dn/dWL   -- fund: {crossing_TM02['dn_dWL_fund']:.6e}/nm   TM02: {crossing_TM02['dn_dWL_target']:.6e}/nm")
    print(f"dn/dwidth -- fund: {fund_grad_TM02['dn_dwidth']:.6e}/nm   TM02: {target_grad_TM02['dn_dwidth']:.6e}/nm")
    print(f"dn/dheight-- fund: {fund_grad_TM02['dn_dheight']:.6e}/nm   TM02: {target_grad_TM02['dn_dheight']:.6e}/nm")
    print(f"d(PM wavelength)/d(width):  {wl_grad_TM02['dWL_dwidth']:.6f} nm per nm")
    print(f"d(PM wavelength)/d(height): {wl_grad_TM02['dWL_dheight']:.6f} nm per nm")

TM00 vs TM02: no crossing found in range -- excluded from table.


In [ ]:
## === TM00 vs TM40 (mode 21) ===

wavelengths_shg_TM40 = np.arange(210, 251, 2.0)
Nmodes_TM40 = 30  # index 21 + window(5) + buffer -- still cheaper than 35

crossing_TM40 = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg_TM40,
                                  fund_mode_idx=0, target_mode_idx=21,
                                  num_modes_fund=2, num_modes_target=Nmodes_TM40,
                                  x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

if crossing_TM40 is None:
    print(f"No crossing found for TM40 in {wavelengths_shg_TM40[0]}-{wavelengths_shg_TM40[-1]}nm.")
else:
    print(f"TM40 crossing: {crossing_TM40['wavelength_shg']:.3f} nm, n_eff={crossing_TM40['n_eff']:.5f}")
    print(f"dn/dWL -- fund: {crossing_TM40['dn_dWL_fund']:.6e}/nm   TM40: {crossing_TM40['dn_dWL_target']:.6e}/nm")

In [ ]:
if crossing_TM40 is None:
    print("Skipping TM40 gradients -- no crossing found in range.")
    fund_grad_TM40 = None
else:
    fund_wavelength_TM40 = 2 * crossing_TM40['wavelength_shg']
    fund_base_point_TM40 = (h_core, w_core, fund_wavelength_TM40)

    fund_grad_TM40 = eh.mode_gradient_width_height(anisotropic_equation, fund_base_point_TM40, 0, num_modes=6,
                                                    x_resolution=dx/2, y_resolution=dy/2)

    print(f"Fundamental base point: h={h_core}, w={w_core}, wavelength={fund_wavelength_TM40:.3f} nm")
    print(f"dn/dwidth:  {fund_grad_TM40['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {fund_grad_TM40['dn_dheight']:.6e} per nm")

Fundamental base point: h=350.0, w=400.0, wavelength=487.973 nm
dn/dwidth:  2.764670e-04 per nm
dn/dheight: 3.731424e-04 per nm


In [29]:
if crossing_TM40 is None:
    print("Skipping TM40 target gradient -- no crossing found in range.")
    target_grad_TM40 = None
else:
    target_base_point_TM40 = (h_core, w_core, crossing_TM40['wavelength_shg'])

    target_grad_TM40 = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_TM40, 21, num_modes=Nmodes_TM40,
                                                      x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

    print(f"TM40 base point: h={h_core}, w={w_core}, wavelength={crossing_TM40['wavelength_shg']:.3f} nm")
    print(f"dn/dwidth:  {target_grad_TM40['dn_dwidth']:.6e} per nm")
    print(f"dn/dheight: {target_grad_TM40['dn_dheight']:.6e} per nm")

TM40 base point: h=350.0, w=400.0, wavelength=243.986 nm
dn/dwidth:  4.113122e-04 per nm
dn/dheight: 2.009432e-03 per nm


In [30]:
if crossing_TM40 is None:
    print("TM00 vs TM40: no crossing found in range -- excluded from table.")
    wl_grad_TM40 = None
else:
    wl_grad_TM40 = eh.implicit_wavelength_gradient(fund_grad_TM40, target_grad_TM40, crossing_TM40)

    print("=== TM00 vs TM40 phase-matching table row (h=350, w=400) ===")
    print(f"Phase-matching wavelength (SHG side): {crossing_TM40['wavelength_shg']:.3f} nm")
    print(f"n_eff at crossing: {crossing_TM40['n_eff']:.5f}")
    print(f"dn/dWL   -- fund: {crossing_TM40['dn_dWL_fund']:.6e}/nm   TM40: {crossing_TM40['dn_dWL_target']:.6e}/nm")
    print(f"dn/dwidth -- fund: {fund_grad_TM40['dn_dwidth']:.6e}/nm   TM40: {target_grad_TM40['dn_dwidth']:.6e}/nm")
    print(f"dn/dheight-- fund: {fund_grad_TM40['dn_dheight']:.6e}/nm   TM40: {target_grad_TM40['dn_dheight']:.6e}/nm")
    print(f"d(PM wavelength)/d(width):  {wl_grad_TM40['dWL_dwidth']:.6f} nm per nm")
    print(f"d(PM wavelength)/d(height): {wl_grad_TM40['dWL_dheight']:.6f} nm per nm")

=== TM00 vs TM40 phase-matching table row (h=350, w=400) ===
Phase-matching wavelength (SHG side): 243.986 nm
n_eff at crossing: 2.02796
dn/dWL   -- fund: -1.368928e-03/nm   TM40: -9.152151e-03/nm
dn/dwidth -- fund: 2.764670e-04/nm   TM40: 4.113122e-04/nm
dn/dheight-- fund: 3.731424e-04/nm   TM40: 2.009432e-03/nm
d(PM wavelength)/d(width):  0.017325 nm per nm
d(PM wavelength)/d(height): 0.210233 nm per nm


## Full table: all four mode pairs (h=350, w=400)

In [31]:
def _row(name, idx, crossing, fund_grad, target_grad, wl_grad):
    if crossing is None:
        return None
    return {
        'target_mode_name': name, 'target_mode_idx': idx, 'h_core': h_core, 'w_core': w_core,
        'wavelength_shg': crossing['wavelength_shg'], 'n_eff': crossing['n_eff'],
        'dn_dWL_fund': crossing['dn_dWL_fund'], 'dn_dWL_target': crossing['dn_dWL_target'],
        'dn_dwidth_fund': fund_grad['dn_dwidth'], 'dn_dwidth_target': target_grad['dn_dwidth'],
        'dn_dheight_fund': fund_grad['dn_dheight'], 'dn_dheight_target': target_grad['dn_dheight'],
        'dWL_dwidth': wl_grad['dWL_dwidth'], 'dWL_dheight': wl_grad['dWL_dheight'],
    }

pairs = [
    ('TM04', 30, crossing_h350, fund_grad_h350, target_grad_h350, wl_grad_h350),
    ('TM40', 21, crossing_TM40, fund_grad_TM40, target_grad_TM40, wl_grad_TM40),
    ('TM02', 6, crossing_TM02, fund_grad_TM02, target_grad_TM02, wl_grad_TM02),
    ('TM20', 4, crossing_TM20, fund_grad_TM20, target_grad_TM20, wl_grad_TM20),
]

all_rows = [_row(*p) for p in pairs]

for name, idx, crossing, *_ in pairs:
    if crossing is None:
        print(f"Note: TM00 vs {name} excluded -- no crossing found in the 210-250nm range of interest.")

print()
print(eh.format_table(all_rows))

Note: TM00 vs TM02 excluded -- no crossing found in the 210-250nm range of interest.
Note: TM00 vs TM20 excluded -- no crossing found in the 210-250nm range of interest.

Mode         | h            | w            | PM WL (nm)   | n_eff        | dWL/dw       | dWL/dh      
------------------------------------------------------------------------------------------------------
TM04         | 350.0        | 400.0        | 228.244      | 2.05082      | -0.011926    | 0.211742    
TM40         | 350.0        | 400.0        | 243.986      | 2.02796      | 0.017325     | 0.210233    


## Diagnostic: re-check TM40's gradient with overlap visibility

Independent COMSOL work says TM40's phase-matching wavelength should be width-sensitive, not
height-sensitive -- the opposite of what we just measured. `mode_gradient_width_height` now returns
the overlap for each of its 4 internal perturbation points, so we can see directly whether the
width or height (or both) evaluations lost track of the true TM40 mode. Re-running the full target
gradient (same cost as before, ~15-20 min at num_modes=30) since overlap wasn't captured last time.

In [32]:
import importlib
importlib.reload(eh)

diag_TM40 = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_TM40, 21, num_modes=Nmodes_TM40,
                                           x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print(f"dn/dwidth:  {diag_TM40['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {diag_TM40['dn_dheight']:.6e} per nm\n")

for key in ['w_plus', 'w_minus', 'h_plus', 'h_minus']:
    r = diag_TM40[key]
    flag = "  <-- LOW OVERLAP, likely tracked a different mode!" if r['overlap'] < 0.8 else ""
    print(f"{key:8s}: mode_idx={r['mode_idx']:3d}, overlap={r['overlap']:.4f}, n_eff={r['n_eff']:.5f}{flag}")

dn/dwidth:  4.113122e-04 per nm
dn/dheight: 2.009432e-03 per nm

w_plus  : mode_idx= 21, overlap=0.9779, n_eff=2.02993
w_minus : mode_idx= 21, overlap=0.9917, n_eff=2.02582
h_plus  : mode_idx= 21, overlap=0.9604, n_eff=2.03780
h_minus : mode_idx= 22, overlap=0.9847, n_eff=2.01770


## Quick visual double-check: mode 21 vs mode 22 at h=345 (the crossing point)

Before spending more time on the re-derivation, confirm visually which mode actually looks like
TM40 (4 lobes along width) right at the height where the tracker found the index swap. If mode 21
still clearly looks like TM40 and mode 22 looks like something else, the tracking was correct and
the crossing explanation holds. If it's ambiguous or reversed, that's a red flag worth stopping for.

In [33]:
em = eh.launch_session('tm40_crossing_check', verbose=True)
eh.setup_waveguide(em, anisotropic_equation, 345.0, w_core)  # h=345 -- the h_minus point from the diagnostic
eh.solve_modes(em, 'anchor', wavelength=target_base_point_TM40[2], num_modes=Nmodes_TM40,
               x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

em.plot(mode=21, component='Ey')

'ok'

In [34]:
em.plot(mode=22, component='Ey')

'ok'

## Re-derive TM40's height gradient with a smaller step

`h_plus` tracked mode 21, `h_minus` tracked mode 22 -- both high overlap, meaning TM40 undergoes a
real near-degeneracy/avoided crossing with a neighboring mode somewhere between h=345 and h=350.
The original `d_height=5.0` central difference straddled that crossing, likely inflating the
measured `dn/dheight`. Retrying with `d_height=1.0` to stay on one side of it. Same cost as before
(~15-20 min, still num_modes=30).

In [35]:
diag_TM40_small_dh = eh.mode_gradient_width_height(anisotropic_equation, target_base_point_TM40, 21, num_modes=Nmodes_TM40,
                                                    d_height=1.0, x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print(f"dn/dwidth:  {diag_TM40_small_dh['dn_dwidth']:.6e} per nm")
print(f"dn/dheight: {diag_TM40_small_dh['dn_dheight']:.6e} per nm  (was {diag_TM40['dn_dheight']:.6e} with d_height=5.0)\n")

for key in ['w_plus', 'w_minus', 'h_plus', 'h_minus']:
    r = diag_TM40_small_dh[key]
    flag = "  <-- LOW OVERLAP" if r['overlap'] < 0.8 else ""
    print(f"{key:8s}: mode_idx={r['mode_idx']:3d}, overlap={r['overlap']:.4f}, n_eff={r['n_eff']:.5f}{flag}")

if diag_TM40_small_dh['h_plus']['mode_idx'] != diag_TM40_small_dh['h_minus']['mode_idx']:
    print("\nStill straddling the crossing at d_height=1.0 -- may need an even smaller step, or the")
    print("crossing sits very close to h=350 itself.")

dn/dwidth:  4.113122e-04 per nm
dn/dheight: 2.167509e-03 per nm  (was 2.009432e-03 with d_height=5.0)

w_plus  : mode_idx= 21, overlap=0.9779, n_eff=2.02993
w_minus : mode_idx= 21, overlap=0.9917, n_eff=2.02582
h_plus  : mode_idx= 21, overlap=0.9918, n_eff=2.03065
h_minus : mode_idx= 21, overlap=0.9999, n_eff=2.02631


## Overnight batch: TM04 at width = 250, 350, 500, 700nm (h=350 fixed) + linear/quadratic fit

Completes the width spot-check plan (250/350/500/700) at the real h=350 anchor -- our only prior
width=350 check was at the h=335 stand-in, not here. These are big jumps from w=400, so: wider
tracking window (mode index may shift more than the Â±1-2 seen before), wider crossing search range
(in case the crossing itself moves outside 210-250nm), and a try/except per width so one
failure/no-crossing doesn't stop the rest. Uses only `solve_and_identify`/`find_crossing` -- not
`mode_gradient_width_height`, which is still under investigation for the TM40 discrepancy.

**Two overlap checks per width**, not just one: (1) at the initial tracking step (anchor wavelength
228.244nm), and (2) re-verified via `solve_and_identify` again at the actual crossing wavelength
`find_crossing` locates -- which can be quite different from the tracking wavelength for these big
jumps, so a mode confidently identified at one wavelength isn't guaranteed to still be it at the
other without checking. `sweep()`'s internal continuity (which `find_crossing` relies on to hold
the mode steady across its own wavelength grid) has only been validated near the w=400 anchor, not
at these much wider geometries.

Also fits both a linear and quadratic polynomial to all collected (width, wavelength) points
(including the w=400 anchor) -- gives an empirical answer to "how much curvature is there really"
without needing new analytical Hessian code tonight. Safe to run unattended overnight.

In [36]:
overnight_widths = [250, 350, 500, 700]
overnight_results = {}

anchor_h = h_core            # 350, already the real target height
anchor_w = w_core            # 400
anchor_mode_idx_TM04 = 30
anchor_wl = crossing_h350['wavelength_shg']  # 228.244

for w in overnight_widths:
    print(f"\n=== TM04 spot-check at width={w}nm ===")
    try:
        id_result = eh.solve_and_identify(anisotropic_equation, (anchor_h, anchor_w, anchor_wl), anchor_mode_idx_TM04,
                                           (anchor_h, w, anchor_wl), num_modes=35, window=10,
                                           x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
        print(f"  [Check 1/2] Tracked at anchor WL={anchor_wl:.3f}nm: mode index {id_result['mode_idx']}, overlap={id_result['overlap']:.4f}")
        if id_result['overlap'] < 0.8:
            print(f"  WARNING: low overlap - tracking may be unreliable at this width.")

        wavelengths_wide = np.arange(190, 271, 2.0)
        crossing_w = eh.find_crossing(anisotropic_equation, anchor_h, w, wavelengths_wide,
                                       fund_mode_idx=0, target_mode_idx=id_result['mode_idx'],
                                       num_modes_fund=2, num_modes_target=35,
                                       x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
        if crossing_w is None:
            print(f"  No crossing found in {wavelengths_wide[0]}-{wavelengths_wide[-1]}nm.")
            overnight_results[w] = None
            continue

        # Check 2: re-verify mode identity AT the actual crossing wavelength, not just the
        # tracking wavelength above -- sweep()'s internal continuity across its own wavelength
        # grid has only been validated near the w=400 anchor, not at these wider geometries.
        recheck = eh.solve_and_identify(anisotropic_equation, (anchor_h, anchor_w, anchor_wl), anchor_mode_idx_TM04,
                                         (anchor_h, w, crossing_w['wavelength_shg']), num_modes=35, window=10,
                                         x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)
        print(f"  [Check 2/2] Re-verified at crossing WL={crossing_w['wavelength_shg']:.3f}nm: mode index {recheck['mode_idx']}, overlap={recheck['overlap']:.4f}")
        if recheck['overlap'] < 0.8:
            print(f"  WARNING: low overlap at crossing WL - result may not actually be TM04.")
        if recheck['mode_idx'] != id_result['mode_idx']:
            print(f"  WARNING: mode index disagrees between the two checks ({id_result['mode_idx']} vs {recheck['mode_idx']}) -- crossing may have been found for the wrong mode.")

        predicted_wl = anchor_wl + wl_grad_h350['dWL_dwidth'] * (w - anchor_w)
        diff = crossing_w['wavelength_shg'] - predicted_wl
        print(f"  Brute-force crossing: {crossing_w['wavelength_shg']:.3f} nm")
        print(f"  Linear model prediction: {predicted_wl:.3f} nm")
        print(f"  Difference: {diff:.3f} nm")
        overnight_results[w] = {'mode_idx': id_result['mode_idx'], 'overlap': id_result['overlap'],
                                 'recheck_overlap': recheck['overlap'], 'recheck_mode_idx': recheck['mode_idx'],
                                 'crossing': crossing_w, 'predicted_wl': predicted_wl, 'diff': diff}
    except Exception as e:
        print(f"  FAILED at width {w}: {repr(e)}")
        overnight_results[w] = None

print("\n\n=== Summary: TM04 linear model spot-checks ===")
print(f"{'Width':<8}| {'Mode idx':<9}| {'Overlap1':<9}| {'Overlap2':<9}| {'Actual WL':<11}| {'Predicted':<11}| {'Diff':<8}")
for w in overnight_widths:
    r = overnight_results.get(w)
    if r is None:
        print(f"{w:<8}| FAILED/no crossing")
    else:
        flag = "" if r['mode_idx'] == r['recheck_mode_idx'] else "  <-- MODE MISMATCH"
        print(f"{w:<8}| {r['mode_idx']:<9}| {r['overlap']:<9.4f}| {r['recheck_overlap']:<9.4f}| {r['crossing']['wavelength_shg']:<11.3f}| {r['predicted_wl']:<11.3f}| {r['diff']:<8.3f}{flag}")

# --- Linear vs quadratic polynomial fit across all collected points (including the w=400 anchor) ---
# Only use points where both overlap checks were confident (>= 0.8) and mode index agreed --
# a shaky point would silently corrupt the polynomial fit otherwise.
trusted_widths = [w for w in overnight_widths if overnight_results.get(w) is not None
                   and overnight_results[w]['overlap'] >= 0.8 and overnight_results[w]['recheck_overlap'] >= 0.8
                   and overnight_results[w]['mode_idx'] == overnight_results[w]['recheck_mode_idx']]
untrusted_widths = [w for w in overnight_widths if w not in trusted_widths and overnight_results.get(w) is not None]
if untrusted_widths:
    print(f"\nExcluding from fit (failed overlap/mode-agreement checks): {untrusted_widths}")

fit_widths = [anchor_w] + trusted_widths
fit_wavelengths = [anchor_wl] + [overnight_results[w]['crossing']['wavelength_shg'] for w in trusted_widths]

print(f"\n=== Linear vs quadratic fit ({len(fit_widths)} trusted points) ===")
if len(fit_widths) < 3:
    print("Not enough trusted points to fit a quadratic (need >= 3).")
else:
    order = np.argsort(fit_widths)
    fit_widths = np.array(fit_widths)[order]
    fit_wavelengths = np.array(fit_wavelengths)[order]
    print("Points used (width, wavelength):", list(zip(fit_widths, fit_wavelengths)))

    lin_coeffs = np.polyfit(fit_widths, fit_wavelengths, 1)
    quad_coeffs = np.polyfit(fit_widths, fit_wavelengths, 2)
    print(f"\nLinear fit:    WL = {lin_coeffs[0]:.6e}*w + {lin_coeffs[1]:.4f}")
    print(f"Quadratic fit: WL = {quad_coeffs[0]:.6e}*w^2 + {quad_coeffs[1]:.6e}*w + {quad_coeffs[2]:.4f}")
    print(f"(compare quadratic's linear-order coefficient, {quad_coeffs[1]:.6e}, against the analytical dWL_dwidth={wl_grad_h350['dWL_dwidth']:.6e} at w=400)")

    lin_resid = fit_wavelengths - np.polyval(lin_coeffs, fit_widths)
    quad_resid = fit_wavelengths - np.polyval(quad_coeffs, fit_widths)
    print(f"\nLinear fit residuals:    {lin_resid}")
    print(f"Quadratic fit residuals: {quad_resid}")
    print(f"Max |linear residual|:    {np.max(np.abs(lin_resid)):.4f} nm")
    print(f"Max |quadratic residual|: {np.max(np.abs(quad_resid)):.4f} nm")


=== TM04 spot-check at width=250nm ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 21, overlap=0.0304
  [Check 2/2] Re-verified at crossing WL=226.091nm: mode index 21, overlap=0.0268
  Brute-force crossing: 226.091 nm
  Linear model prediction: 230.033 nm
  Difference: -3.942 nm

=== TM04 spot-check at width=350nm ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 28, overlap=0.8741
  [Check 2/2] Re-verified at crossing WL=228.818nm: mode index 28, overlap=0.8677
  Brute-force crossing: 228.818 nm
  Linear model prediction: 228.840 nm
  Difference: -0.022 nm

=== TM04 spot-check at width=500nm ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 30, overlap=0.0013
  [Check 2/2] Re-verified at crossing WL=237.688nm: mode index 30, overlap=0.0015
  Brute-force crossing: 237.688 nm
  Linear model prediction: 227.052 nm
  Difference: 10.636 nm

=== TM04 spot-check at width=700nm ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 30, overlap=0.0002

## Recompute TM40's wl_grad with the clean (crossing-free) height derivative

Pure arithmetic, no EMode calls -- combines the already-computed `fund_grad_TM40` and `crossing_TM40`
with the clean `d_height=1.0` result from `diag_TM40_small_dh`, instead of the crossing-contaminated
`d_height=5.0` value used in the original `target_grad_TM40`/`wl_grad_TM40`.

In [37]:
target_grad_TM40_clean = {'dn_dwidth': diag_TM40_small_dh['dn_dwidth'], 'dn_dheight': diag_TM40_small_dh['dn_dheight']}
wl_grad_TM40_clean = eh.implicit_wavelength_gradient(fund_grad_TM40, target_grad_TM40_clean, crossing_TM40)

print("=== TM00 vs TM40 phase-matching table row (h=350, w=400) -- UPDATED with clean height derivative ===")
print(f"Phase-matching wavelength (SHG side): {crossing_TM40['wavelength_shg']:.3f} nm")
print(f"n_eff at crossing: {crossing_TM40['n_eff']:.5f}")
print(f"dn/dWL   -- fund: {crossing_TM40['dn_dWL_fund']:.6e}/nm   TM40: {crossing_TM40['dn_dWL_target']:.6e}/nm")
print(f"dn/dwidth -- fund: {fund_grad_TM40['dn_dwidth']:.6e}/nm   TM40: {target_grad_TM40_clean['dn_dwidth']:.6e}/nm")
print(f"dn/dheight-- fund: {fund_grad_TM40['dn_dheight']:.6e}/nm   TM40: {target_grad_TM40_clean['dn_dheight']:.6e}/nm  (was {target_grad_TM40['dn_dheight']:.6e})")
print(f"d(PM wavelength)/d(width):  {wl_grad_TM40_clean['dWL_dwidth']:.6f} nm per nm  (unchanged from before)")
print(f"d(PM wavelength)/d(height): {wl_grad_TM40_clean['dWL_dheight']:.6f} nm per nm  (was {wl_grad_TM40['dWL_dheight']:.6f})")

=== TM00 vs TM40 phase-matching table row (h=350, w=400) -- UPDATED with clean height derivative ===
Phase-matching wavelength (SHG side): 243.986 nm
n_eff at crossing: 2.02796
dn/dWL   -- fund: -1.368928e-03/nm   TM40: -9.152151e-03/nm
dn/dwidth -- fund: 2.764670e-04/nm   TM40: 4.113122e-04/nm
dn/dheight-- fund: 3.731424e-04/nm   TM40: 2.167509e-03/nm  (was 2.009432e-03)
d(PM wavelength)/d(width):  0.017325 nm per nm  (unchanged from before)
d(PM wavelength)/d(height): 0.230543 nm per nm  (was 0.210233)


## Retry TM04 at w=300/700 with wider search (num_modes=50, window=20)

Quick test before committing to the full h/w grid: does a bigger num_modes + wider overlap window
recover tracking at widths where the overnight batch (num_modes=35, window=10) collapsed to
near-zero overlap? Only 2 points for now -- if this works, we scale up to the full plan.

In [38]:
import importlib
importlib.reload(eh)

anchor_point_TM04 = (h_core, w_core, crossing_h350['wavelength_shg'])  # (350, 400, 228.244)
wavelengths_retry = np.arange(190, 281, 2.0)

retry_targets = [(h_core, 300), (h_core, 700)]
retry_results_TM04 = eh.sweep_crossing(anisotropic_equation, anchor_point_TM04, 30, retry_targets,
                                        wavelengths_retry, num_modes=50, window=20,
                                        x_resolution=dx/2, y_resolution=dy/2, max_effective_index=2.6)

print("\n=== Retry summary ===")
for key, r in retry_results_TM04.items():
    if r is None:
        print(f"{key}: FAILED/no crossing")
    else:
        print(f"{key}: mode_idx={r['mode_idx']}, overlap={r['overlap']:.4f}, recheck_overlap={r['recheck_overlap']:.4f}, crossing={r['crossing']['wavelength_shg']:.3f}nm, trusted={r['trusted']}")



=== Point h=350.0, w=300 ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 23, overlap=0.5802
  [Check 2/2] Re-verified at crossing WL=229.825nm: mode index 23, overlap=0.6383
  Crossing: 229.825 nm, n_eff=2.01156

=== Point h=350.0, w=700 ===
  [Check 1/2] Tracked at anchor WL=228.244nm: mode index 49, overlap=0.0033
  [Check 2/2] Re-verified at crossing WL=230.820nm: mode index 49, overlap=0.0035
  Crossing: 230.820 nm, n_eff=2.08442

=== Retry summary ===
(350.0, 300): mode_idx=23, overlap=0.5802, recheck_overlap=0.6383, crossing=229.825nm, trusted=False
(350.0, 700): mode_idx=49, overlap=0.0033, recheck_overlap=0.0035, crossing=230.820nm, trusted=False


## TM40: visually confirm mode identity at every gradient-stencil point (one cell per point)

Split into separate cells -- one per geometry point -- so there's no ambiguity about which plots
belong to which point. Each plots modes 21/22/23 (now that mode 23 looks like the real TM40, not
21). Run them one at a time, in order.

In [3]:
import importlib
importlib.reload(eh)

wl_TM40 = 243.986  # crossing_TM40['wavelength_shg'], already computed earlier -- avoids re-running find_crossing
h0, w0 = h_core, w_core  # 350, 400

### Point: anchor (h=350, w=400)

In [6]:
Nmodes_TM40 = 35  
h, w = h0, w0
print(f"=== anchor: h={h}, w={w} ===")
em = eh.launch_session('tm40_mode_id_check', verbose=False)
eh.setup_waveguide(em, anisotropic_equation, h, w)
eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=Nmodes_TM40,
               x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
for mode in (21, 22, 23):
    print(f"  -- mode {mode} --")
    em.plot(mode=mode, component='Ey')
em.close(save=False)

=== anchor: h=350.0, w=400.0 ===
  -- mode 21 --
  -- mode 22 --
  -- mode 23 --


### Point: w_minus (h=350, w=350)

In [9]:
h, w = h0, w0 - 50
print(f"=== w_minus: h={h}, w={w} ===")
em = eh.launch_session('tm40_mode_id_check', verbose=False)
eh.setup_waveguide(em, anisotropic_equation, h, w)
eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=Nmodes_TM40,
               x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
print(f"  -- mode {23} --")
em.plot(mode=mode, component='Ey')
em.close(save=False)

=== w_minus: h=350.0, w=350.0 ===
  -- mode 23 --


### Point: w_plus (h=350, w=450)

In [10]:
h, w = h0, w0 + 50
print(f"=== w_plus: h={h}, w={w} ===")
em = eh.launch_session('tm40_mode_id_check', verbose=False)
eh.setup_waveguide(em, anisotropic_equation, h, w)
eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=Nmodes_TM40,
               x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
for mode in (21, 22, 23):
    print(f"  -- mode {mode} --")
    em.plot(mode=mode, component='Ey')
em.close(save=False)

=== w_plus: h=350.0, w=450.0 ===
  -- mode 21 --
  -- mode 22 --
  -- mode 23 --


### Point: h_minus (h=335, w=400)

In [11]:
h, w = h0 - 15, w0
print(f"=== h_minus: h={h}, w={w} ===")
em = eh.launch_session('tm40_mode_id_check', verbose=False)
eh.setup_waveguide(em, anisotropic_equation, h, w)
eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=Nmodes_TM40,
               x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
for mode in (21, 22, 23):
    print(f"  -- mode {mode} --")
    em.plot(mode=mode, component='Ey')
em.close(save=False)

=== h_minus: h=335.0, w=400.0 ===
  -- mode 21 --
  -- mode 22 --
  -- mode 23 --


### Point: h_plus (h=365, w=400)  -- NOTE: was mislabeled h=355; h0+15=365, not h0+5

In [12]:
h, w = h0 + 15, w0
print(f"=== h_plus: h={h}, w={w} ===")  # actually h=365 (h0+15), not 355
em = eh.launch_session('tm40_mode_id_check', verbose=False)
eh.setup_waveguide(em, anisotropic_equation, h, w)
eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=Nmodes_TM40,
               x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
for mode in (21, 22, 23):
    print(f"  -- mode {mode} --")
    em.plot(mode=mode, component='Ey')
em.close(save=False)

=== h_plus: h=365.0, w=400.0 ===
  -- mode 21 --
  -- mode 22 --
  -- mode 23 --


## TM40: direct dn/dwidth and dn/dheight from confirmed mode indices (no tracking needed)

Every mode index at every stencil point is now visually confirmed (ground truth), so the secant
slopes can be read directly off `n_eff` at the known-correct index -- no `track_mode`/`overlap()`
guessing, no window-size questions. Uses the confirmed indices: w_minus=26, w_plus=19, h_minus=22,
h_plus=23 (at h=365, not the originally-labeled 355 -- see the fixed h_plus cell above).

In [13]:
import importlib
importlib.reload(eh)

wl_TM40 = 243.986

confirmed_points_TM40 = {
    'w_minus': (350.0, 350.0, 26),
    'w_plus':  (350.0, 450.0, 19),
    'h_minus': (335.0, 400.0, 22),
    'h_plus':  (365.0, 400.0, 23),
}

n_eff_TM40 = {}
for label, (h, w, mode_idx) in confirmed_points_TM40.items():
    em = eh.launch_session('tm40_direct_neff', verbose=False)
    eh.setup_waveguide(em, anisotropic_equation, h, w)
    fdm_result = eh.solve_modes(em, 'check', wavelength=wl_TM40, num_modes=30,
                                 x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
    n_eff = fdm_result['n_eff_tilde'][mode_idx].real
    n_eff_TM40[label] = n_eff
    print(f"{label}: h={h}, w={w}, mode={mode_idx}, n_eff={n_eff:.6f}")
    em.close(save=False)

dn_dwidth_TM40_direct = (n_eff_TM40['w_plus'] - n_eff_TM40['w_minus']) / (450.0 - 350.0)
dn_dheight_TM40_direct = (n_eff_TM40['h_plus'] - n_eff_TM40['h_minus']) / (365.0 - 335.0)

print(f"\ndn/dwidth  (direct, confirmed-mode) = {dn_dwidth_TM40_direct:.6e} per nm"
      f"   (COMSOL: ~1.4-2e-3/nm; old mislabeled-mode EMode value: 4.113e-4/nm)")
print(f"dn/dheight (direct, confirmed-mode) = {dn_dheight_TM40_direct:.6e} per nm"
      f"   (COMSOL: ~3.0e-5/nm;    old mislabeled-mode EMode value: 2.168e-3/nm)")

w_minus: h=350.0, w=350.0, mode=26, n_eff=1.867188
w_plus: h=350.0, w=450.0, mode=19, n_eff=2.107903
h_minus: h=335.0, w=400.0, mode=22, n_eff=2.007188
h_plus: h=365.0, w=400.0, mode=23, n_eff=2.008208

dn/dwidth  (direct, confirmed-mode) = 2.407151e-03 per nm   (COMSOL: ~1.4-2e-3/nm; old mislabeled-mode EMode value: 4.113e-4/nm)
dn/dheight (direct, confirmed-mode) = 3.398116e-05 per nm   (COMSOL: ~3.0e-5/nm;    old mislabeled-mode EMode value: 2.168e-3/nm)


## TM40: redo crossing search with the CORRECT mode index (23, not 21)

`crossing_TM40` (243.986nm) was found searching for where the fundamental crosses mode 21 (TM13),
not the real TM40 (mode 23). Redo with `target_mode_idx=23` to get the true crossing wavelength,
then rebuild `target_grad_TM40_correct` from the already-confirmed dn/dwidth and dn/dheight (no new
EMode calls needed there), and recompute the final `wl_grad_TM40`.

In [ ]:
import importlib
importlib.reload(eh)
wavelengths_shg_TM40 = np.arange(210, 251, 2.0)

crossing_TM40_correct = eh.find_crossing(anisotropic_equation, h_core, w_core, wavelengths_shg_TM40,
                                          fund_mode_idx=0, target_mode_idx=23,
                                          num_modes_fund=2, num_modes_target=Nmodes_TM40,
                                          x_resolution=dx, y_resolution=dy, max_effective_index=2.6)
print(f"Corrected TM40 crossing: {crossing_TM40_correct['wavelength_shg']:.3f} nm, n_eff={crossing_TM40_correct['n_eff']:.5f}")
print(f"(was 243.986 nm with the wrong mode 21)")

# Already have these from the direct-n_eff cell above -- no new EMode calls needed.
target_grad_TM40_correct = {'dn_dwidth': 2.407151e-03, 'dn_dheight': 3.398116e-05}
fund_grad_TM40 = {'dn_dwidth': 2.764670e-04, 'dn_dheight': 3.731424e-04}

# fund_grad_TM40 was computed at 2x the (wrong) crossing wavelength -- only recompute if the
# corrected crossing wavelength has shifted enough to matter (fundamental varies slowly/smoothly,
# so a small shift likely won't change fund_grad meaningfully; check the printed wavelength above).
wl_grad_TM40_correct = eh.implicit_wavelength_gradient(fund_grad_TM40, target_grad_TM40_correct, crossing_TM40_correct)

print(f"\ndn/dwidth  -- fund: {fund_grad_TM40['dn_dwidth']:.6e}/nm   TM40: {target_grad_TM40_correct['dn_dwidth']:.6e}/nm")
print(f"dn/dheight -- fund: {fund_grad_TM40['dn_dheight']:.6e}/nm   TM40: {target_grad_TM40_correct['dn_dheight']:.6e}/nm")
print(f"d(PM wavelength)/d(width):  {wl_grad_TM40_correct['dWL_dwidth']:.6f} nm per nm   (COMSOL: ~0.24 nm/nm; old wrong-mode value: 0.0173)")
print(f"d(PM wavelength)/d(height): {wl_grad_TM40_correct['dWL_dheight']:.6f} nm per nm   (old wrong-mode value: 0.2305)")

Corrected TM40 crossing: 241.672 nm, n_eff=2.03125
(was 243.986 nm with the wrong mode 21)


NameError: name 'fund_grad_TM40' is not defined

In [16]:
fund_grad_TM40 = {'dn_dwidth': 2.764670e-04, 'dn_dheight': 3.731424e-04}

# fund_grad_TM40 was computed at 2x the (wrong) crossing wavelength -- only recompute if the
# corrected crossing wavelength has shifted enough to matter (fundamental varies slowly/smoothly,
# so a small shift likely won't change fund_grad meaningfully; check the printed wavelength above).
wl_grad_TM40_correct = eh.implicit_wavelength_gradient(fund_grad_TM40, target_grad_TM40_correct, crossing_TM40_correct)

print(f"\ndn/dwidth  -- fund: {fund_grad_TM40['dn_dwidth']:.6e}/nm   TM40: {target_grad_TM40_correct['dn_dwidth']:.6e}/nm")
print(f"dn/dheight -- fund: {fund_grad_TM40['dn_dheight']:.6e}/nm   TM40: {target_grad_TM40_correct['dn_dheight']:.6e}/nm")
print(f"d(PM wavelength)/d(width):  {wl_grad_TM40_correct['dWL_dwidth']:.6f} nm per nm   (COMSOL: ~0.24 nm/nm; old wrong-mode value: 0.0173)")
print(f"d(PM wavelength)/d(height): {wl_grad_TM40_correct['dWL_dheight']:.6f} nm per nm   (old wrong-mode value: 0.2305)")


dn/dwidth  -- fund: 2.764670e-04/nm   TM40: 2.407151e-03/nm
dn/dheight -- fund: 3.731424e-04/nm   TM40: 3.398116e-05/nm
d(PM wavelength)/d(width):  0.240897 nm per nm   (COMSOL: ~0.24 nm/nm; old wrong-mode value: 0.0173)
d(PM wavelength)/d(height): -0.038346 nm per nm   (old wrong-mode value: 0.2305)


## TM40: large-step dn/dwidth check, and a sidewall-angle hypothesis

The COMSOL data (own crossing-wavelength calc, no EMode) gives TM_40 a `dWL_dwidth` of ~+0.24
nm/nm, flat across w=300-600nm -- vs. our own +0.0173 nm/nm, a ~14x gap that isn't explained by
COMSOL's slope being curved (it isn't, in this range). TM_04 matches almost exactly (-0.012 to
-0.014 COMSOL vs -0.0119 ours), so the pipeline/method itself checks out; this looks like a real
TM_40-specific disagreement.

Two things to rule out cheaply, from the existing h=350,w=400,mode=21 anchor:
1. **Step size**: redo dn/dwidth with a 100nm-wide stencil (w=350/450, matching COMSOL's grid)
   instead of the existing +/-5nm one. COMSOL's own slope is flat over this range, so if ours
   doesn't move much either, step size isn't the explanation.
2. **Sidewall angle**: the user's COMSOL sidewall angle for this dataset may have been ~89deg from
   horizontal, not the 85deg (`sidewall_angle=5` in our convention) we've been assuming to match.
   TM_40 (4 lobes across width) is plausibly far more sensitive to the exact sidewall profile than
   TM_04 (uniform across width) -- which would also explain why TM_04 matched so well while TM_40
   didn't. Re-run at `sidewall_angle=1` (~89deg from horizontal) alongside the current default of 5.

In [ ]:
import importlib
importlib.reload(eh)

wl_TM40 = crossing_TM40['wavelength_shg']
anchor_TM40 = (h_core, w_core, wl_TM40)  # (350, 400, 243.986)

sidewall_results_TM40 = {}
for sidewall_angle in (5, 1):
    w_plus = eh.solve_and_identify(anisotropic_equation, anchor_TM40, 21, (h_core, w_core + 50, wl_TM40),
                                    num_modes=Nmodes_TM40, window=5, x_resolution=dx, y_resolution=dy,
                                    sidewall_angle=sidewall_angle)
    w_minus = eh.solve_and_identify(anisotropic_equation, anchor_TM40, 21, (h_core, w_core - 50, wl_TM40),
                                     num_modes=Nmodes_TM40, window=5, x_resolution=dx, y_resolution=dy,
                                     sidewall_angle=sidewall_angle)
    dn_dwidth_large = (w_plus['n_eff'] - w_minus['n_eff']) / 100.0
    sidewall_results_TM40[sidewall_angle] = {
        'w_plus': w_plus, 'w_minus': w_minus, 'dn_dwidth_large_step': dn_dwidth_large,
    }
    print(f"sidewall_angle={sidewall_angle}:")
    print(f"  w=450: mode_idx={w_plus['mode_idx']}, overlap={w_plus['overlap']:.4f}, n_eff={w_plus['n_eff']:.5f}")
    print(f"  w=350: mode_idx={w_minus['mode_idx']}, overlap={w_minus['overlap']:.4f}, n_eff={w_minus['n_eff']:.5f}")
    print(f"  dn/dwidth (100nm step) = {dn_dwidth_large:.6e} per nm  (was 4.113e-4/nm with +/-5nm step, sidewall_angle=5)\n")